In [ ]:
from pyi18next.i18next import I18next
from adaptation.misc import NameAnonymizer
from utils.graph_builder import LocalizationGraphProcessor, traverse_namespaces
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from controllers.encoders.jaccard import JaccardEncoder
from services.calibrator_factory import CalibratorFactory
from controllers.retrievers.retriever import Retriever
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.vector_numpy import cosine_similarity
from core.settings import get_settings
import pandas as pd
import numpy as np
import time
import os


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} bert={} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/spanish_word2vec/word2vec.bin'} spacy={'es': 'es_core_news_sm'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/lstm_mean_cosine_noaug'} allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000


In [3]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "final")
structure_dir = os.path.join(localization_dir,  "structure")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [4]:
namespaces = traverse_namespaces(language_dir, languages)


In [5]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [6]:
name_anonymizer = NameAnonymizer(
	names_path=spanish_names_path,
	whitelist_path=name_whitelist_path,
	replacement="[UNK]"
)


In [7]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer("sbert")
model_registry.build_spacy()
# model_registry.build_word2vec()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
manager = MultilingualManager(
	encoder_factory=encoder_factory,
	calibrator_factory=calibrator_factory, 
	name_anonymizer=name_anonymizer,
	base_dir=database_dir
)
model_types = model_registry.active_model_types()


2026-06-28 04:33:57.503 | DEBUG    | services.model_registry:_create_loader:59 - Registering sbert loader for 'es'
2026-06-28 04:33:57.504 | DEBUG    | services.model_registry:_create_loader:59 - Registering spaCy loader for 'es'
2026-06-28 04:33:57.504 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-06-28 04:33:58.544 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-06-28 04:33:58.544 | DEBUG    | services.lazy_loader:model:16 - Loading spaCy for 'es'...
2026-06-28 04:33:58.894 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded spaCy for 'es'


In [8]:
language = next(iter(settings.languages))

retrievers: list[Retriever] = [
    manager.get_dense_retriever(
        language=language,
        model_type="jaccard",
        similarity_fn=JaccardEncoder.jaccard,
    ),
    manager.get_dense_retriever(
        language=language,
        model_type="tfidf",
        similarity_fn=cosine_similarity,
    ),
    manager.get_dense_retriever(
        language=language,
        model_type="sbert",
        similarity_fn=cosine_similarity,
    ),
]


In [9]:
class LeaveOneOutEvaluator:
	def __init__(self, retriever: Retriever):
		self.retriever = retriever

	def _collect_predictions(self, corpus: list[str], metadata: list[dict]):
		predictions = []

		total_time = 0
		for i, query in enumerate(corpus):
			train_corpus = corpus[:i] + corpus[i + 1:]
			train_metadata = metadata[:i] + metadata[i + 1:]

			self.retriever.fit(train_corpus)

			start = time.perf_counter()

			indices, scores, _ = self.retriever.search(
				query,
				top_k=1
			)

			total_time += time.perf_counter() - start

			if len(indices) <= 0:
				predictions.append({
					"score": 0,
					"correct": False
				})
				continue


			predicted_index = indices[0]
			
			predicted_branch = train_metadata[predicted_index]["group_index"]

			true_branch = metadata[i]["group_index"]

			predictions.append({
				"score": float(scores[0]),
				"correct": predicted_branch == true_branch
			})

		avg_time_ms = total_time / len(corpus) * 1000

		return predictions, avg_time_ms

	def evaluate(self, corpus: list[str], metadata: list[dict], thresholds: np.ndarray):
		predictions, avg_time_ms = self._collect_predictions(
			corpus,
			metadata
		)
		
		threshold_results = {}

		for threshold in thresholds:
			results = self._calculate_metrics(predictions, threshold)
			threshold_results[float(threshold)] = results

		# Mejor umbral según f1
		best_threshold = max(
			threshold_results,
			key=lambda t: threshold_results[t]["f1"]
		)

		return {
			"best_threshold": best_threshold,
			"avg_time_ms": avg_time_ms,
			**threshold_results[best_threshold],
			"thresholds": threshold_results
		}

	def _calculate_metrics(self, predictions: list[dict], threshold: float):
		correct_accept = 0  # Rama correcta + encima del umbral (TP)
		wrong_accept = 0    # Rama incorrecta + encima del umbral (FP)
		correct_reject = 0  # Rama correcta + debajo del umbral (FN)
		wrong_reject = 0    # Rama incorrecta + debajo del umbral (no es TN porque todos los ejemplos pertenecen a una rama)

		total = len(predictions)

		for prediction in predictions:
			accepted = prediction["score"] >= threshold
			correct = prediction["correct"]

			if accepted and correct:
				correct_accept += 1

			elif accepted and not correct:
				wrong_accept += 1

			elif not accepted and correct:
				correct_reject += 1

			else:
				wrong_reject += 1
		
		# El porcentaje de consultas correctas
		branch_accuracy = self._safe_div(
			correct_accept,
			total
		)

		# Cuando el sistema elige una rama, cúantas veces es la correcta
		# TP = TP + FP
		precision = self._safe_div(
			correct_accept,
			correct_accept + wrong_accept
		)

		# Cuando el sistema encuentra la rama correcta, cúantas veces se supera el threshold
		# TP = TP + FN
		recall = self._safe_div(
			correct_accept,
			correct_accept + correct_reject
		)

		f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

		return {
			"branch_accuracy": branch_accuracy,
			"precision": precision,
			"recall": recall,
			"f1": f1,

			"correct_accept": correct_accept,
			"wrong_accept": wrong_accept,
			"correct_reject": correct_reject,
			"wrong_reject": wrong_reject,
		}


	def _safe_div(self, a, b):
		return a / b if b else 0
	

In [10]:
class SimilarityEvaluationRunner(LocalizationGraphProcessor):
	def __init__(self, i18n: I18next, languages: set[str], base_dir: str, retrievers: list[Retriever], min_examples: int = 3, min_groups: int = 2):
		super().__init__(i18n, languages, base_dir)

		self.results = []
		self.situations = []

		self.retrievers = retrievers

		self.methods = set(
			retriever.encoder.name
			for retriever in self.retrievers
		)

		self.min_examples = min_examples
		self.min_groups = min_groups
	
	def process_similarity(self, responses: list[dict], node_key: str, language: str):
		corpus = []
		metadata = []

		valid_groups = []

		for group_idx, group in enumerate(responses):
			expanded_texts = []

			for text in group["text"]:
				new_texts = self.process_data(text)
				expanded_texts.extend(new_texts)

			if len(expanded_texts) >= self.min_examples:
				valid_groups.append((group_idx, expanded_texts))

		num_groups = len(valid_groups)

		self.situations.append({
			"node": node_key,
			"num_groups": num_groups,
			"num_examples": sum(
				len(texts)
				for _, texts in valid_groups
			),
			"evaluated": num_groups >= self.min_groups
		})

		if num_groups >= self.min_groups:
			idx = 0
			for group_idx, expanded_texts in valid_groups:
				for sentence_idx, text in enumerate(expanded_texts):
					corpus.append(text)

					metadata.append({
						"index": idx,
						"text": text,
						"group_index": group_idx,
						"sentence_index": sentence_idx,
						"node": node_key,
					})

					idx += 1

			# Evaluación
			for retriever in self.retrievers:				
				evaluator = LeaveOneOutEvaluator(retriever)

				result = evaluator.evaluate(
					corpus=corpus,
					metadata=metadata,
					thresholds=np.arange(0.5, 1.0, 0.05)
				)

				self.results.append({
					"node": node_key,
					"method": retriever.encoder.name,
					**result,
				})

	def get_method_results(self):
		df = (
			pd.DataFrame(self.results)
			.groupby("method", as_index=False)
			.agg({
				"best_threshold": "mean",
				"branch_accuracy": "mean",
				"precision": "mean",
				"recall": "mean",
				"f1": "mean",
				"avg_time_ms": "mean",
			})
		)

		return df
	
	def get_situation_results(self, metric: str = "f1"):
		df = pd.DataFrame(self.results)

		# Filas -> situaciones
		# Columnas -> modelos
		# Valores -> métrica
		pivot_df = (
			df.pivot(
				index="node",
				columns="method",
				values=metric
			)
			# Ordenar las situaciones alftabéticamente
			.sort_index()
		)

		pivot_df.loc["avg"] = pivot_df.mean()
		pivot_df.loc["std"] = pivot_df.std()
		
		# Renombrar eje de las filas
		return pivot_df.rename_axis("situation")
	
	def get_situation_statistics(self):
		return pd.DataFrame(self.situations)
		

In [11]:
runner = SimilarityEvaluationRunner(
	i18n=i18n,
	languages=languages,
	base_dir=structure_dir,
    retrievers=retrievers
)


In [12]:
runner.run()


Total visited nodes: 665


In [13]:
method = runner.get_method_results()
display(method)


,method,best_threshold,branch_accuracy,precision,recall,f1,avg_time_ms
0,jaccard,0.50,0.000000,0.000000,0.0,0.000000,0.100950
1,sbert,0.55,0.847826,0.866667,1.0,0.928571,8.948378
2,tfidf,0.50,0.000000,0.000000,0.0,0.000000,0.154226


In [14]:
situation = runner.get_situation_results()
display(situation)


method,jaccard,sbert,tfidf
situation,,,
scene1Classroom_part2_thanks2,0.0,0.928571,0.0
avg,0.0,0.928571,0.0
std,0.0,0.000000,0.0


In [15]:
stat = runner.get_situation_statistics()
display(stat)


,node,num_groups,num_examples,evaluated
0,scene1Classroom_part2_thanks2,2,46,True
